In [5]:
import pandas as pd
import os
from datetime import datetime
from sklearn.datasets import fetch_20newsgroups

def analyze_dataset(csv_path, save_report=True, dataset_name=None):
    """
    Анализирует CSV датасет и выводит информацию о нем.
    
    Args:
        csv_path: путь к CSV файлу или список путей
        save_report: сохранять ли отчет в файл
        dataset_name: имя датасета (если None - из имени файла)
    
    Returns:
        DataFrame с данными
    """
    # Если передан список путей - объединяем
    if isinstance(csv_path, list):
        dfs = []
        for path in csv_path:
            if os.path.exists(path):
                dfs.append(pd.read_csv(path))
            else:
                print(f"❌ Файл {path} не найден!")
        
        if not dfs:
            return None
        
        df = pd.concat(dfs, ignore_index=True)
        
        if dataset_name is None:
            dataset_name = os.path.basename(csv_path[0]).replace('.csv', '').replace('train_', '').replace('test_', '')
        file_path = csv_path[0]
    else:
        if not os.path.exists(csv_path):
            print(f"❌ Файл {csv_path} не найден!")
            return None
        
        df = pd.read_csv(csv_path)
        
        if dataset_name is None:
            dataset_name = os.path.basename(csv_path).replace('.csv', '')
        file_path = csv_path
    
    # Собираем отчет
    report = []
    report.append("=" * 70)
    report.append(f"📊 ИНФОРМАЦИЯ О ДАТАСЕТЕ: {dataset_name}")
    report.append("=" * 70)
    report.append(f"📅 Дата анализа: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    if isinstance(csv_path, list):
        report.append(f"📄 Файлы: {', '.join(csv_path)}")
    else:
        report.append(f"📄 Файл: {csv_path}")
        report.append(f"📏 Размер: {os.path.getsize(csv_path) / 1024 / 1024:.2f} МБ")
    
    report.append(f"📊 Строк: {len(df)}")
    report.append(f"📋 Колонок: {len(df.columns)}")
    report.append(f"📋 Колонки: {', '.join(df.columns.tolist())}")
    
    # Пропуски
    report.append(f"\n🔍 Пропуски в данных:")
    nulls = df.isnull().sum()
    if nulls.sum() == 0:
        report.append("  ✅ Пропусков нет")
    else:
        for col, count in nulls.items():
            if count > 0:
                report.append(f"  {col}: {count}")
    
    # Категории/метки
    label_cols = [c for c in df.columns if c.lower() in ['category', 'label', 'sentiment', 'class', 'class index', 'target']]
    for col in label_cols:
        report.append(f"\n📂 Распределение по '{col}':")
        value_counts = df[col].value_counts()
        for val, count in value_counts.head(15).items():
            report.append(f"  {str(val):<30} → {count}")
        if len(value_counts) > 15:
            report.append(f"  ... и еще {len(value_counts) - 15} категорий")
        report.append(f"  Всего категорий: {len(value_counts)}")
    
    # Текстовые колонки
    text_cols = [c for c in df.columns if c.lower() in ['content', 'text', 'title', 'review', 'description', 'body']]
    for col in text_cols:
        if col in df.columns:
            df[f'{col}_len'] = df[col].astype(str).str.len()
            report.append(f"\n📝 Статистика длины '{col}':")
            report.append(f"  Средняя: {df[f'{col}_len'].mean():.0f} символов")
            report.append(f"  Мин: {df[f'{col}_len'].min()}")
            report.append(f"  Макс: {df[f'{col}_len'].max()}")
            report.append(f"  Медиана: {df[f'{col}_len'].median():.0f}")
    
    # Примеры
    report.append(f"\n📄 Примеры записей (первые 3):")
    report.append("-" * 70)
    for i in range(min(3, len(df))):
        row = df.iloc[i]
        report.append(f"\n[{i+1}]")
        for col in df.columns:
            val = str(row[col])
            report.append(f"  {col}: {val}")
    
    report.append("\n" + "=" * 70)
    
    # Выводим и сохраняем
    report_text = "\n".join(report)
    print(report_text)
    
    if save_report:
        if isinstance(csv_path, list):
            report_dir = os.path.dirname(csv_path[0])
        else:
            report_dir = os.path.dirname(csv_path)
        report_file = os.path.join(report_dir, f"{dataset_name}_report.txt")
        with open(report_file, "w", encoding="utf-8") as f:
            f.write(report_text)
        print(f"\n💾 Отчет сохранен: {report_file}")
    
    return df


def analyze_imdb(csv_path="../data/raw/imdb/imdb.csv", save_report=True):
    """Анализирует датасет IMDB"""
    if not os.path.exists(csv_path):
        print(f"❌ Файл {csv_path} не найден!")
        return None
    
    df = pd.read_csv(csv_path)
    
    report = []
    report.append("=" * 70)
    report.append("📊 ИНФОРМАЦИЯ О ДАТАСЕТЕ: IMDB")
    report.append("=" * 70)
    report.append(f"📅 Дата анализа: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"📄 Файл: {csv_path}")
    report.append(f"📏 Размер: {os.path.getsize(csv_path) / 1024 / 1024:.2f} МБ")
    report.append(f"📊 Строк: {len(df)}")
    report.append(f"📋 Колонки: {', '.join(df.columns.tolist())}")
    
    # Пропуски
    report.append(f"\n🔍 Пропуски в данных:")
    nulls = df.isnull().sum()
    if nulls.sum() == 0:
        report.append("  ✅ Пропусков нет")
    else:
        for col, count in nulls.items():
            if count > 0:
                report.append(f"  {col}: {count}")
    
    # Распределение по sentiment
    if 'sentiment' in df.columns:
        report.append(f"\n📂 Распределение по sentiment:")
        sent_counts = df['sentiment'].value_counts()
        for val, count in sent_counts.items():
            emoji = "😊" if str(val).lower() == "positive" else "😞" if str(val).lower() == "negative" else ""
            report.append(f"  {str(val):<15} {emoji} → {count}")
    
    # Длина текста
    text_col = 'review'
    if text_col in df.columns:
        df['text_len'] = df[text_col].astype(str).str.len()
        report.append(f"\n📝 Статистика длины отзывов:")
        report.append(f"  Средняя: {df['text_len'].mean():.0f} символов")
        report.append(f"  Мин: {df['text_len'].min()}")
        report.append(f"  Макс: {df['text_len'].max()}")
        report.append(f"  Медиана: {df['text_len'].median():.0f}")
    
    # Примеры
    report.append(f"\n📄 Примеры записей:")
    report.append("-" * 70)
    for label in df['sentiment'].unique():
        subset = df[df['sentiment'] == label].head(2)
        for i, (_, row) in enumerate(subset.iterrows()):
            report.append(f"\n[{label.upper()}] Пример {i+1}:")
            review = str(row.get('review', ''))
            report.append(f"  {review}...")
    
    report.append("\n" + "=" * 70)
    
    report_text = "\n".join(report)
    print(report_text)
    
    if save_report:
        report_dir = os.path.dirname(csv_path)
        report_file = os.path.join(report_dir, "imdb_report.txt")
        with open(report_file, "w", encoding="utf-8") as f:
            f.write(report_text)
        print(f"\n💾 Отчет сохранен: {report_file}")
    
    return df


def analyze_20newsgroups(save_report=True, subset='all'):
    """Анализирует датасет 20 Newsgroups"""
    print("\n" + "=" * 70)
    print("📊 ЗАГРУЗКА 20 NEWSGROUPS")
    print("=" * 70)
    
    if subset == 'all':
        datasets = {}
        for name in ['train', 'test']:
            print(f"  Загрузка {name}...")
            bunch = fetch_20newsgroups(subset=name, remove=('headers', 'footers', 'quotes'))
            datasets[name] = pd.DataFrame({
                'text': bunch.data,
                'label': bunch.target,
                'category': [bunch.target_names[t] for t in bunch.target]
            })
        
        df = pd.concat([datasets['train'], datasets['test']], ignore_index=True)
        print(f"  ✅ Загружено: train={len(datasets['train'])}, test={len(datasets['test'])}")
    else:
        print(f"  Загрузка {subset}...")
        bunch = fetch_20newsgroups(subset=subset, remove=('headers', 'footers', 'quotes'))
        df = pd.DataFrame({
            'text': bunch.data,
            'label': bunch.target,
            'category': [bunch.target_names[t] for t in bunch.target]
        })
        print(f"  ✅ Загружено: {len(df)}")
    
    df['text_len'] = df['text'].str.len()
    
    report = []
    report.append("=" * 70)
    report.append("📊 ИНФОРМАЦИЯ О ДАТАСЕТЕ: 20 Newsgroups")
    report.append("=" * 70)
    report.append(f"📅 Дата анализа: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"📊 Строк: {len(df)}")
    report.append(f"📋 Колонок: {len(df.columns)}")
    report.append(f"📋 Колонки: {', '.join(df.columns.tolist())}")
    
    report.append(f"\n🔍 Пропуски в данных:")
    report.append("  ✅ Пропусков нет")
    
    report.append(f"\n📂 Распределение по категориям (20 классов):")
    cat_counts = df['category'].value_counts()
    for cat, count in cat_counts.items():
        report.append(f"  {cat:<30} → {count}")
    
    report.append(f"\n📝 Статистика длины текста:")
    report.append(f"  Средняя: {df['text_len'].mean():.0f} символов")
    report.append(f"  Мин: {df['text_len'].min()}")
    report.append(f"  Макс: {df['text_len'].max()}")
    report.append(f"  Медиана: {df['text_len'].median():.0f}")
    
    report.append(f"\n📊 Классы (label):")
    report.append(f"  Всего классов: {df['label'].nunique()}")
    report.append(f"  Диапазон: {df['label'].min()} - {df['label'].max()}")
    
    report.append("\n" + "=" * 70)
    
    report_text = "\n".join(report)
    print(report_text)
    
    if save_report:
        report_file = "20newsgroups_report.txt"
        with open(report_file, "w", encoding="utf-8") as f:
            f.write(report_text)
        print(f"\n💾 Отчет сохранен: {report_file}")
    
    return df


def analyze_all_datasets():
    """Анализирует все датасеты и сохраняет отчеты"""
    datasets = {}
    
    # Udmurt Media
    print("\n" + "🔹" * 35)
    print("📰 UDMURT MEDIA")
    print("🔹" * 35)
    datasets['udmurt'] = analyze_dataset("../data/raw/udmurt_media/udmurt_media.csv")
    
    # AG News (train + test)
    print("\n" + "🔹" * 35)
    print("📰 AG NEWS")
    print("🔹" * 35)
    datasets['ag_news'] = analyze_dataset(
        ["../data/raw/ag_news/train.csv", "../data/raw/ag_news/test.csv"],
        dataset_name="ag_news"
    )
    
    # IMDB
    print("\n" + "🔹" * 35)
    print("🎬 IMDB")
    print("🔹" * 35)
    datasets['imdb'] = analyze_imdb("../data/raw/imdb/imdb.csv")
    
    # 20 Newsgroups
    print("\n" + "🔹" * 35)
    print("📧 20 NEWSGROUPS")
    print("🔹" * 35)
    datasets['20news'] = analyze_20newsgroups()
    
    # Сводная таблица
    print("\n" + "=" * 70)
    print("📊 СВОДКА ПО ВСЕМ ДАТАСЕТАМ")
    print("=" * 70)
    print(f"{'Датасет':<15} {'Строк':<10} {'Классов':<10} {'Ср. длина':<12} {'Размер'}")
    print("-" * 70)
    
    for name, df in datasets.items():
        if df is not None:
            rows = len(df)
            
            # Количество классов
            if 'category' in df.columns:
                classes = df['category'].nunique()
            elif 'label' in df.columns:
                classes = df['label'].nunique()
            elif 'sentiment' in df.columns:
                classes = df['sentiment'].nunique()
            else:
                classes = 0
            
            # Средняя длина текста
            text_col = next((c for c in ['content', 'text', 'review'] if c in df.columns), None)
            avg_len = df[text_col].astype(str).str.len().mean() if text_col else 0
            
            # Размер файла
            if name == 'udmurt':
                path = "../data/raw/udmurt_media/udmurt_media.csv"
                size = f"{os.path.getsize(path) / 1024 / 1024:.1f} MB" if os.path.exists(path) else "N/A"
            elif name == 'ag_news':
                paths = ["../data/raw/ag_news/train.csv", "../data/raw/ag_news/test.csv"]
                size = sum(os.path.getsize(p) for p in paths if os.path.exists(p)) / 1024 / 1024
                size = f"{size:.1f} MB"
            elif name == 'imdb':
                path = "../data/raw/imdb/imdb.csv"
                size = f"{os.path.getsize(path) / 1024 / 1024:.1f} MB" if os.path.exists(path) else "N/A"
            elif name == '20news':
                size = "in memory"
            else:
                size = "N/A"
            
            print(f"{name:<15} {rows:<10} {classes:<10} {avg_len:<12.0f} {size}")
    
    print("=" * 70)
    
    return datasets


# =====================================================
# 🚀 Запуск
# =====================================================
if __name__ == "__main__":
    datasets = analyze_all_datasets()


🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
📰 UDMURT MEDIA
🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
📊 ИНФОРМАЦИЯ О ДАТАСЕТЕ: udmurt_media
📅 Дата анализа: 2026-05-04 23:18:31
📄 Файл: ../data/raw/udmurt_media/udmurt_media.csv
📏 Размер: 6.74 МБ
📊 Строк: 3718
📋 Колонок: 3
📋 Колонки: title, content, category

🔍 Пропуски в данных:
  ✅ Пропусков нет

📂 Распределение по 'category':
  Мерлык                         → 287
  Спорт                          → 287
  Авто                           → 287
  Лулчеберет но туризм           → 286
  Политика                       → 286
  Шимучыръёс                     → 286
  Лэсьтӥськон                    → 286
  Тазалык                        → 286
  ЖКХ                            → 286
  Инкуазь                        → 286
  Экономика но коньдон           → 285
  Дышетон но тодос               → 285
  Гурт возёс                     → 284
  Ужбергатон                     → 1
  Всего категорий: 14

📝 Статистика длины 'title':
  Средняя: 68 символов
  Мин: 24
  Макс:

In [14]:
import sys
sys.path.append('../src')
from data_loader import get_udmurt_media
texts, labels = get_udmurt_media()

print("✅ Загружено:", len(texts))
print("🏷️ Меток:", len(set(labels)))
# print("✅ Данные загружены")
# print(f"📄 Всего строк: {len(df)}")
# print(f"🏷️ Категорий: {len(categories)}")


📊 Categories mapping:
0: Авто
1: Гурт возёс
2: Дышетон но тодос
3: ЖКХ
4: Инкуазь
5: Лулчеберет но туризм
6: Лэсьтӥськон
7: Мерлык
8: Политика
9: Спорт
10: Тазалык
11: Шимучыръёс
12: Экономика но коньдон
✅ Загружено: 3718
🏷️ Меток: 13


In [16]:
print("📊 Categories mapping:\n")

for i, cat in enumerate(labels):
    print(f"{i}: {cat}")

📊 Categories mapping:

0: 5
1: 8
2: 11
3: 12
4: 7
5: 6
6: 2
7: 9
8: 0
9: 10
10: 3
11: 4
12: 1
13: 5
14: 8
15: 11
16: 12
17: 7
18: 6
19: 2
20: 9
21: 5
22: 10
23: 3
24: 0
25: 4
26: 1
27: 8
28: 11
29: 12
30: 7
31: 6
32: 2
33: 9
34: 5
35: 3
36: 10
37: 0
38: 4
39: 1
40: 8
41: 11
42: 12
43: 7
44: 6
45: 2
46: 9
47: 5
48: 3
49: 0
50: 10
51: 4
52: 1
53: 8
54: 11
55: 12
56: 7
57: 6
58: 2
59: 9
60: 5
61: 3
62: 0
63: 10
64: 4
65: 1
66: 8
67: 11
68: 12
69: 7
70: 6
71: 2
72: 9
73: 5
74: 3
75: 0
76: 10
77: 4
78: 1
79: 8
80: 11
81: 12
82: 7
83: 6
84: 2
85: 9
86: 5
87: 3
88: 0
89: 10
90: 4
91: 1
92: 8
93: 11
94: 12
95: 7
96: 6
97: 2
98: 9
99: 5
100: 3
101: 0
102: 10
103: 4
104: 1
105: 8
106: 11
107: 12
108: 7
109: 6
110: 2
111: 9
112: 5
113: 3
114: 0
115: 10
116: 4
117: 1
118: 8
119: 11
120: 12
121: 6
122: 7
123: 2
124: 9
125: 5
126: 3
127: 0
128: 10
129: 4
130: 1
131: 8
132: 11
133: 12
134: 6
135: 7
136: 2
137: 9
138: 5
139: 3
140: 0
141: 10
142: 1
143: 4
144: 8
145: 11
146: 12
147: 6
148: 7
149: 2
15

In [17]:
for i in range(3):
    print(f"\n--- SAMPLE {i+1} ---")
    print("LABEL:", labels[i])
    print("TEXT:", texts[i])


--- SAMPLE 1 ---
LABEL: 5
TEXT: Владимир Путин Удмуртиысь бесерманэн Надежда Сидороваен но нылыныз пумиськиз Россилэн азьмуртэз Владимир Путин «Знание» огазеяськонлэн «Знание. Первые» марафоназ кунысь ӧжыт лыдъем выжы калыкъёсысь огмуртъёсын, марафонэ пыриськисьёсын, пумиськиз. Соос лулчеберетэз, кылэз, ӧнеръёсты утён ужпумъёс сярысь вераськизы. Пӧлазы Балезино ёросысь Юнда черкогуртысь бесерман Надежда Сидорова но вал. Ужрадэ со 14 аресъем Злата нылыныз вуиз. Со сярысь элькун тӧролэн но кивалтэтлэн пресс-службазы ивортӥз. Надежда Сидорова – трос нылпиё анай. Со «Азвесь крезь» калыккылос ансамблен но «Чингыли» нылпи клубен кивалтэ, калык гуръёсты, эктонъёсты, фольклорез улӟытонэн выре. Нылъёсызлэн атайзы Алексей нимысьтыз ожгар ужрадын быриз. Злата нылыз 8-тӥ классын дышетске, калыккылос клубе пыриське. Со сям-йылолъёсты эскере, бесерман кылэз дышетэ, вашкала крезьёсты быдэсъя, Удмуртиын но солэн сьӧраз пӧртэм ужрадъёсы пыриськылэ. Надежда Сидорова нылыныз Владимир Путин но пумиськонэ